# Workshop: reduce and higher-order functions

### Wednesday, February 28, 2024

In today's workshop we'll continue our discussion of functional programming, picking up where we left off on Monday. We worked with map and filter; now let's look at reduce.
We'll also play around a bit with lambda expressions and "fancier" functional programming patterns.

In [3]:
import functools # We need this for reduce!

## 1.) warming up with lambda expressions

Let's start by making sure we understand lambda expressions.
Recall that lambda expressions allow us to define functions in a much more concise manner compared to writing a function definition statement.
They tend to look like `lambda [args] : [expression]`. This line of code evaluates to a <i>function</i>. That means we can write things like `f = lambda [args] : [expression]`, and then we can call `f` like a function.


In [4]:
f = lambda x, y, z : x*y*z

def myfn(x, y, z):
    return x*y*z

In [5]:
f

<function __main__.<lambda>(x, y, z)>

In [6]:
f( 1,2,3)

6

In [7]:
myfn(1,2,3)

6

In [8]:
lambda x, y, z : x*y*z # This evaluates to a function!

<function __main__.<lambda>(x, y, z)>

The next few miniature exercises will get you some practice with lambda expressions.

### 1.1

Write a lambda expression for a function that takes a two arguments and returns their sum. Save that lambda expression in a variable `fadd` and verify that you can write things like `fadd(1,2)`, `fadd('cat','dog')`, etc.

In [9]:
fadd = lambda arg1, arg2 : arg1 + arg2
print(fadd(1,2))
print(fadd('cat', 'dog'))

3
catdog


### 1.2

Write a lambda expression for a function that takes a single argument (assumed to be a string) and returns that string with all characters changed to upper case.
Save that lambda expression in a variable `toupper` and verify that, for example, `toupper('AbCDefG')` produces `'ABCDEFG'`.

In [10]:
toupper = lambda arg : arg.upper()
print(toupper('cat'))

CAT


### 1.3

Write a lambda expression that takes two arguments and returns `True` if the first of the two arguments is strictly larger than the second and returns `False` otherwise.
Store this lambda expression in a variable `greaterthan` and verify that, for example, `greaterthan(2,1)` returns `True` while `greaterthan(1,1)` and `greaterthan(0,10)` both return `False`.

In [11]:
greaterthan = lambda arg1, arg2 : arg1 > arg2
print(greaterthan(2,1))
print(greaterthan(1,1))
print(greaterthan(0,10))

True
False
False


## Problem 2: Reduce operations

Let's apply our lambda expressions and make sure we understand the Reduce pattern. A reduce pattern involves:

- a function $f(x,y)$
- an initial value $a_0$ for the accumulator

Then, given a list of elements $x_1,x_2,\dots,x_n$, we initialize an accumulator $A = a_0$ and repeatedly perform the update
$$ a_{t+1} = f( a_t, x_{t+1} ) $$
until we obtain $a_n$.

As a concrete example, reducing the function $f(x,y) = x+y$ (i.e., our `fadd` function from Problem 1) over the list `[1,2,3]` with initial value `0` should produce the result `6`.

This procedure is handled in Python by the `reduce` function in the `itertools` module, but to make sure we understand this idea, let's implement our own.

Write a function `my_reduce` that takes three arguments:

- A function `f` that takes two arguments and returns a single object (we are being vague about types here because we want to be as general as possible)
- An iterable `t` (i.e., `t` supports `for x in t:...`)
- An initial value `a0`, which is the value that the accumulator gets initialized to.

`my_reduce` should return the result of reducing the function `f` along the iterable `t` with initial accumulator value `a0`.

As a test case, you should be able to combine `my_reduce` with the summation lambda expression from the previous problem to compute the sum of the list `[1,2,3]` and to turn the list `['b', 'u', 'c', 'k', 'y']` into the string `'bucky'`.

In [12]:
def my_reduce( f, t, a0 ):
    result = a0
    for i in t:
        result = f(result, i)
    return result

In [13]:
my_reduce( fadd, [1,2,3], 0 ) # Should evaluate to 6.

6

In [14]:
my_reduce( fadd, [1,1,1,1,1,1], 0 ) # Should evaluate to 6.

6

In [15]:
# Should evaluate to 'bucky'
my_reduce( fadd, ['b', 'u', 'c', 'k', 'y'], '')

'bucky'

Now, let's use our `my_reduce` function to write our own version of the `any` operation.
Recall that `any` takes a list `t` as its only argument and returns `True` if one or more entries evaluates to `True` (i.e., if <i>any</i> entry is true) and returns `False` otherwise.

Define a function `my_any` that operates the same as `any`. You should be able to write this function in one line (not counting the definition header, of course) by using `my_reduce`.

As a test case, you should be able to verify that `my_any( t ) == any(t)` for any list `t`.

In [25]:
def my_any( t ):
    for i in t:
        if i:
            return True
    return False

In [26]:
t = [False,False,False]
my_any( t ) == any( t ) # Should evaluate to True.

True

## Problem 3: Putting it all together

Adapted from an exercise in Mary Rose Cook's <i>Practical Introduction to Functional Programming</i> (available at https://maryrosecook.com/blog/post/a-practical-introduction-to-functional-programming).

An important point of the MapReduce framework is that it permits us to avoid iterating over lists with for-loops and similar operations.
List comprehensions, maps, etc. tend to be faster than for-loops in Python.
This preference for avoiding for-loops holds all the moreso in `pandas` and `numpy`, so it's worth getting used to writing in this style.

Try rewriting the following code using map, reduce and filter.

In [27]:
people = [{'name': 'Mary', 'height': 160},
          {'name': 'Isla', 'height': 80},
          {'name': 'Sam'}]

height_total = 0
height_count = 0
for person in people:
    if 'height' in person:
        height_total += person['height']
        height_count += 1

average_height = height_total / height_count # We're assuming that height_count > 0.

print( average_height )
# Should evaluate to 120

120.0


In [34]:
people = [{'name': 'Mary', 'height': 160},
          {'name': 'Isla', 'height': 80},
          {'name': 'Sam'}]

average_height = lambda t : sum(t)/len(t)
average_height([people[0]['height'], people[1]['height']])

120.0

<b>Challenge:</b> write a reduce statement that computes the average in a single pass over the list.

<b>Hint:</b> write a reduce statement that works on pairs (totalheight, count), THEN compute totalheight/count to get the average. I've written such a function for you below.

In [37]:
#f( x, y ) has x is a tuple of the form (totalheight, count) (this is your accumulator)
#and y is a height.
def height_fn( tup, h ):
    # Ideally, we should do some error checking to make sure tup is a tuple of length 2
    # and h is a numeric, but this is just demo code.
    # tup = (totalheight, count)
    return (tup[0]+h, tup[1]+1)

# and now we can reduce this function along the sequence of heights.

## Problem 4: Higher-order functions and pipelines

Adapted from a running example in Mary Rose Cook's <i>Practical Introduction to Functional Programming</i> (available at https://maryrosecook.com/blog/post/a-practical-introduction-to-functional-programming).

Consider the following "database" of mathematicians, whose names are inconsistently formatted and whose countries of origin are incorrect (they are all of Polish origin):

In [38]:
mathematicians = [{'name': 'stanislaw. ULAM', 'country': 'France', 'deceased': True},
              {'name': 'andrew odlyzko', 'country': 'Poland', 'deceased': False},
              {'name': 'Aleksander. Rajchman.', 'country': 'Belarus', 'deceased': True},
              {'name': 'jerzy neyman', 'country': 'Germany', 'deceased': True} ]

The "non-functional" way to correct this "database" is to do something like the following:

In [39]:
def format_mathematicians(mat):
    for m in mat:
        m['country'] = 'Poland'
        m['name'] = m['name'].replace('.', '')
        m['name'] = m['name'].title() # Change to Title Case.
        
format_mathematicians( mathematicians )

for m in mathematicians:
    print( m )

{'name': 'Stanislaw Ulam', 'country': 'Poland', 'deceased': True}
{'name': 'Andrew Odlyzko', 'country': 'Poland', 'deceased': False}
{'name': 'Aleksander Rajchman', 'country': 'Poland', 'deceased': True}
{'name': 'Jerzy Neyman', 'country': 'Poland', 'deceased': True}


Now, what would be the more functional approach to this?
Well, the basic idea is that we should have a series of functions, each of which does one particular thing (e.g., fixes capitalization, corrects the "country" field, etc.).
We'll then want to apply these functions to each entry in our list (i.e., to each dictionary), with the output of one being sent to the input of another.

In [40]:
# Resetting the database to the original messy/wrong version.
mathematicians = [{'name': 'stanislaw. ULAM', 'country': 'France', 'deceased': True},
              {'name': 'andrew odlyzko', 'country': 'Poland', 'deceased': False},
              {'name': 'Aleksander. Rajchman.', 'country': 'Belarus', 'deceased': True},
              {'name': 'jerzy neyman', 'country': 'Germany', 'deceased': True} ]

What we would like is to be able to write something more like

In [41]:
# Running this block will cause an error because we haven't defined this, yet.
pipeline_each(mathematicians, [set_poland_as_country,
                            strip_punctuation_from_name,
                            capitalize_names])

NameError: name 'pipeline_each' is not defined

That is, the function `pipeline_each` takes the entries in the list `mathematicians` and to each entry, applies the functions `set_poland_as_country`,`strip_punctuation_from_name` and `capitalize_names` to that entry, passing the output of one to the input of the next.

This code has two obvious advantages:

- It is a lot less cryptic than `format_mathematicians`. It is much easier to understand what is about to happen just from reading the code.
- It is far more reusable-- we can change the set of functions to pipeline, change the list from `mathematicians` to something else, etc.

First things first, then, we need the function `pipeline_each`. This function should take a list `t` and a list of functions `fnlist`, and, for each element `e` of `t`, applies each function in the list `fnlist` to that element, in order, passing the output of each function in the list to the input of the second one.
That is, if `fnlist = [f1, f2, f3]`, then element `e` gets mapped to `f1(f2(f3(e)))`.

Let's write this in a not especially functional style, and then we'll write it in a more functional style.

In [42]:
def pipeline_each_nonfunc( t, fnlist ):
    # Error checking should go here, but this is demo code.
    if len( fnlist ) == 0: # Base case: no functions to apply.
        return t
    else: # Take the first function and apply it to the list
        f0 = fnlist[0]
        fnlist_rest = fnlist[1:]
        return pipeline_each_nonfunc( [f0(e) for e in t], fnlist_rest )

Let's test this on a simple example where we know more or less what to expect.

In [43]:
f1 = lambda x : 2*x
f2 = lambda x : x + 1
# So f1(f2(x))=2x + 1.
pipeline_each_nonfunc( [1,2,3], [f1,f2])

[3, 5, 7]

Now, taking our code above as a starting point, write a functional version of this function, called `pipeline_each`.

<b>Hint:</b> line 8 in our definition of `pipeline_each_nonfunc` definition is obviously a map operation. The other piece of the equation is a `reduce` operation.
Reminder: https://docs.python.org/3/library/functools.html#functools.reduce

Don't feel bad if you get stumped-- this is a tricky one! Mary Rose Cook's solution is at the bottom of this notebook, if you need a hint.

In [45]:
def pipeline_each( t, fnlist ):
    # Error checking should go here, but this is demo code.
    if len( fnlist ) == 0: # Base case: no functions to apply.
        return t
    else: # Take the first function and apply it to the list
        f0 = fnlist[0]
        fnlist_rest = fnlist[1:]
        return pipeline_each( [f0(e) for e in t], fnlist_rest )

<b>Bonus exercise:</b> implement functions to perform the corrections discussed above and use our `pipeline_each` function to correct the "database" `mathematicians`.

In [ ]:
# Mary Rose Cook's solution to the pipeline_each problem
def pipeline_each(data, fns):
    return functools.reduce(lambda a, x: map(x, a), fns, data)

In [ ]:
f1 = lambda x : 2*x
f2 = lambda x : x + 1
# So f1(f2(x))=2x + 1.
list(pipeline_each( [1,2,3], [f1,f2]) )